# CPM Scheduling — Quick Start

This notebook walks through the core `trueppm-scheduler` API:
building a project, running the Critical Path Method (CPM), and
reading the results.

**Install**
```bash
pip install trueppm-scheduler
```

Or from the monorepo:
```bash
pip install -e packages/scheduler
```

In [ ]:
from datetime import date, timedelta
from trueppm_scheduler import (
    Calendar, DateRange, Dependency, DependencyType,
    Project, Task, schedule, CyclicDependencyError,
)

## 1. Define the project

We model a simple software release: Design → (Build + Test in parallel) → Deploy.

In [ ]:
# Calendar: Mon–Fri, no holidays (default)
cal = Calendar()  # working_days=0b0011111 (Mon=bit0 .. Sun=bit6)

tasks = [
    Task(id="design",  name="Design",  duration=timedelta(days=5)),
    Task(id="build",   name="Build",   duration=timedelta(days=10)),
    Task(id="test",    name="Test",    duration=timedelta(days=7)),
    Task(id="deploy",  name="Deploy",  duration=timedelta(days=2)),
]

dependencies = [
    Dependency(predecessor_id="design", successor_id="build"),    # FS (default)
    Dependency(predecessor_id="design", successor_id="test"),     # FS
    Dependency(predecessor_id="build",  successor_id="deploy"),   # FS
    Dependency(predecessor_id="test",   successor_id="deploy"),   # FS
]

project = Project(
    id="release-v1",
    name="Release v1.0",
    start_date=date(2026, 4, 1),
    tasks=tasks,
    dependencies=dependencies,
    calendar=cal,
)

## 2. Run CPM

In [ ]:
result = schedule(project)
print(f"Project start:  {result.project_start}")
print(f"Project finish: {result.project_finish}")
print(f"Critical path:  {' → '.join(result.critical_path)}")

## 3. Inspect per-task results

Each task in `result.tasks` has its ES/EF/LS/LF and float populated.

In [ ]:
print(f"{'Task':<10} {'ES':>12} {'EF':>12} {'LS':>12} {'LF':>12} {'Float':>6} {'Critical':>8}")
print("-" * 72)
for t in result.tasks:
    float_days = t.total_float.days
    print(
        f"{t.name:<10} {str(t.early_start):>12} {str(t.early_finish):>12}"
        f" {str(t.late_start):>12} {str(t.late_finish):>12}"
        f" {float_days:>6} {'✓' if t.is_critical else '':>8}"
    )

## 4. Custom calendar: add a holiday

The Good Friday bank holiday shifts the schedule by one working day.

In [ ]:
cal_with_holiday = Calendar(
    exceptions=[
        DateRange(start=date(2026, 4, 3), end=date(2026, 4, 3)),  # Good Friday
    ]
)

project_with_holiday = Project(
    id="release-v1-holiday",
    name="Release v1.0 (with holiday)",
    start_date=date(2026, 4, 1),
    tasks=[
        Task(id=t.id, name=t.name, duration=t.duration) for t in tasks
    ],
    dependencies=[
        Dependency(predecessor_id=d.predecessor_id, successor_id=d.successor_id)
        for d in dependencies
    ],
    calendar=cal_with_holiday,
)

result_holiday = schedule(project_with_holiday)
print(f"Without holiday: {result.project_finish}")
print(f"With holiday:    {result_holiday.project_finish}")

## 5. SS / FF / SF dependency types

Non-FS dependencies model parallel work with constraints.

In [ ]:
# SS with 2-day lag: testing can start 2 days after coding starts
coding = Task(id="code",  name="Code",  duration=timedelta(days=10))
testing = Task(id="test2", name="Test",  duration=timedelta(days=8))

p2 = Project(
    id="p2",
    name="SS example",
    start_date=date(2026, 4, 1),
    tasks=[coding, testing],
    dependencies=[
        Dependency(
            predecessor_id="code",
            successor_id="test2",
            dep_type=DependencyType.SS,
            lag=timedelta(days=2),
        )
    ],
)

r2 = schedule(p2)
for t in r2.tasks:
    print(f"{t.name}: {t.early_start} → {t.early_finish}")

## 6. Cycle detection

In [ ]:
t_a = Task(id="a", name="A", duration=timedelta(days=3))
t_b = Task(id="b", name="B", duration=timedelta(days=3))

p_cycle = Project(
    id="p-cycle",
    name="Cycle example",
    start_date=date(2026, 4, 1),
    tasks=[t_a, t_b],
    dependencies=[
        Dependency(predecessor_id="a", successor_id="b"),
        Dependency(predecessor_id="b", successor_id="a"),  # cycle!
    ],
)

try:
    schedule(p_cycle)
except CyclicDependencyError as e:
    print(f"Caught cycle: {' → '.join(e.cycle)}")

## 7. Serialising a project to JSON

`Project.to_json()` / `Project.from_json()` round-trip all fields.

In [ ]:
import json

json_str = project.to_json(indent=2)
print(json_str[:400], "...")

# Round-trip
project_rt = Project.from_json(json_str)
result_rt = schedule(project_rt)
assert result_rt.project_finish == result.project_finish
print("\nRound-trip OK:", result_rt.project_finish)